# Image Generation with Diffusers

This tutorial will guide you through the basics of using [🤗 Diffusers](https://huggingface.co/docs/diffusers/index) to generate images from text prompts.

# Setup
Before we start, let’s install the required libraries and set up our environment.

In [ ]:
# Install the Hugging Face Diffusers library
%pip install diffusers

# Install Transformers 
%pip install transformers

#### Import dependencies

In [ ]:
import torch
from diffusers import DiffusionPipeline

#### Select device (GPU if available)

In [ ]:
# Set device
if torch.cuda.is_available():
    torch_dtype = torch.bfloat16
    device = "cuda"
else:
    torch_dtype = torch.float32
    device = "cpu"

✅ Now your environment is ready!
In the next step, we’ll load a pre-trained diffusion model and start generating images.

# Generate Images from Text
Now that our pipeline is set up, let’s generate images using text prompts.
We’ll use the [Stable Diffusion XL](https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0) model.

Given the available GPU memory, we use Stable Diffusion XL instead of Qwen-Image for this lab.

In [ ]:
# Load the pipeline
# Load the Stable Diffusion XL pipeline
model_name = "stabilityai/stable-diffusion-xl-base-1.0"
pipe = DiffusionPipeline.from_pretrained(model_name, torch_dtype=torch_dtype)

# Move to GPU or CPU depending on availability
pipe = pipe.to(device)

####  Write a prompt

In [ ]:
# Example text prompt
prompt = """
Two anthropomorphic cats wearing cozy boxing robes and bright gloves,
standing in a boxing ring under a hard spotlight.
Sweat mist drifts in the air, the crowd is blurred in the background with a bokeh effect.
Cinematic style, dramatic lighting, detailed close-up of quick jabs and agile footwork.
"""

#### Choose aspect ratio

In [ ]:
# Generate with different aspect ratios
# Predefined aspect ratios
aspect_ratios = {
    "1:1":   (1328, 1328),
    "16:9":  (1664, 928),
    "9:16":  (928, 1664),
    "4:3":   (1472, 1104),
    "3:4":   (1140, 1472),
    "3:2":   (1584, 1056),
    "2:3":   (1056, 1584),
}

# Pick an aspect ratio (e.g., 16:9)
width, height = aspect_ratios["16:9"]

#### Generate the image

In [ ]:
# Disable cudnn
torch.backends.cudnn.enabled = False

In [ ]:
# Generate an image
image = pipe(
    prompt=prompt,
    width=width,
    height=height,
    num_inference_steps=30,   # number of denoising steps
    guidance_scale=7.0,       # higher = closer to prompt
    generator=torch.Generator(device="cuda").manual_seed(42),  # reproducibility
).images[0]

# Display the image
image

## 💡 Practical Tips  

The **number of steps** influences the generation quality:  

- **10 steps** → a rough sketch — fast but blurry.  
- **30 steps** → clear details — the *sweet spot* for most cases.  
- **60 steps** → sharper fur, lighting, and textures — but slower.  

---

The **guidance_scale (CFG)** controls how strongly the model follows your prompt:  

- **Low scale (≈3)** → more creative freedom (may drift away from prompt).  
- **Medium scale (≈7)** → balanced and natural (usually the best choice).  
- **High scale (≈12)** → very strict, but can look rigid or unnatural.  

---

### 🎯 Extra Tips  

- **Resolution**: Start small (e.g., `512×512`) for faster results. Then upscale if you want higher quality.  
- **Random seeds**: Fixing a seed (`manual_seed(42)`) ensures reproducibility. Change it for different variations of the same prompt.  
- **Style words**: Adding phrases like *“cinematic style,” “dramatic lighting,”* or *“highly detailed”* helps guide the model’s artistic direction.  

---

👉 With these adjustments, you can experiment and fine-tune images until they match your vision.  
